A mix of grid sampling and hashing
Fast training but filesize is big

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import cv2
import pandas as pd
import zipfile
import os
import time
import gc
from PIL import Image
from tqdm.notebook import tqdm
from utils import MetricsEngine, visual_compare  # Assumes your utils file is present
import torch.nn.functional as F

torch.manual_seed(42)
torch.compile(mode="reduce-overhead")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True    # speeds up grid_sample and other conv-like ops
torch.backends.cudnn.deterministic = False
print(f"Using device: {device}")
print(f"Initial VRAM Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

Using device: cuda
Initial VRAM Allocated: 54.45 MB


In [10]:
def load_and_prep_data(image_path, downscale_factor=4):
    """Loads image, generates raw coords, and locks everything to GPU."""
    img = Image.open(image_path).convert('RGB')
    w, h = img.size
    new_w, new_h = w // downscale_factor, h // downscale_factor
    img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
    
    img_np = np.array(img)
    img_norm = img_np / 255.0
    
    # 1. Create Raw Coordinates
    y_coords = np.linspace(-1, 1, new_h)
    x_coords = np.linspace(-1, 1, new_w)
    grid_x, grid_y = np.meshgrid(x_coords, y_coords)
    coords_raw = torch.tensor(np.stack([grid_x.flatten(), grid_y.flatten()], axis=-1), dtype=torch.float32)
    
    # 2. Colors
    colors = torch.tensor(img_norm.reshape(-1, 3), dtype=torch.float32)
    
    # 3. Push EVERYTHING to GPU permanently
    return {
        "coords_raw": coords_raw.to(device),
        "colors": colors.to(device),
        "h": new_h, "w": new_w,
        "num_pixels": new_h * new_w,
        "original_np": img_np
    }

# Load Data (Adjust downscale_factor based on your image)
dataset = load_and_prep_data("jcsmr-1.jpg", downscale_factor=4)
print(f"Resolution: {dataset['w']}x{dataset['h']} | Total Pixels: {dataset['num_pixels']}")
print(f"VRAM after data load: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

Resolution: 1404x936 | Total Pixels: 1314144
VRAM after data load: 64.47 MB


In [11]:
class OptimizedHashGrid2D(nn.Module):
    def __init__(
        self,
        num_levels=12,
        base_res=16,
        max_res=1024,
        feature_dim=2,
        log2_hashmap_size=13
    ):
        super().__init__()
        self.num_levels = num_levels
        self.feature_dim = feature_dim
        self.max_entries = 1 << log2_hashmap_size

        growth_factor = np.exp((np.log(max_res) - np.log(base_res)) / (num_levels - 1))
        self.resolutions = [int(base_res * (growth_factor ** i)) for i in range(num_levels)]

        # ---------- Split levels into dense (grid_sample) and sparse (hash) ----------
        self.dense_grids = nn.ParameterList()   # each: (1, feat, res, res)
        self.hash_tables = nn.ParameterList()   # each: (table_size, feat)
        self.hash_resolutions = []              # resolutions of sparse levels
        self.hash_sizes = []                    # actual table sizes of sparse levels
        self.dense_resolutions = []             # resolutions of dense levels

        for res in self.resolutions:
            table_size = min(res * res, self.max_entries)
            is_dense = (res * res <= self.max_entries)
            if is_dense:
                # Store as 4D tensor ready for grid_sample
                # (flat table -> (1, feat, res, res))
                grid = nn.Parameter(
                    torch.randn(1, feature_dim, res, res) * 0.01
                )
                self.dense_grids.append(grid)
                self.dense_resolutions.append(res)
            else:
                # Sparse hash table
                table = nn.Parameter(torch.randn(table_size, feature_dim) * 0.01)
                self.hash_tables.append(table)
                self.hash_resolutions.append(res)
                self.hash_sizes.append(table_size)

        # Decoder unchanged
        input_dim = num_levels * feature_dim
        self.decoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )

        self.register_buffer("primes", torch.tensor([1, 2654435761], dtype=torch.long))

    def hash_fn(self, coords_int, table_size):
        # coords_int: (N, 2) long
        x = coords_int * self.primes
        h = torch.bitwise_xor(x[..., 0], x[..., 1])
        return h % table_size

    def forward(self, coords):
        coords = (coords + 1) / 2.0
        N = coords.shape[0]
        num_levels = len(self.dense_grids) + len(self.hash_tables)
        out = torch.empty(N, num_levels * self.feature_dim, device=coords.device, dtype=coords.dtype)
        slot = 0

        # Dense levels – unchanged
        for grid, res in zip(self.dense_grids, self.dense_resolutions):
            sg = (2.0 * coords - 1.0).view(1, N, 1, 2)
            f = F.grid_sample(grid, sg, align_corners=True, padding_mode='border')
            out[:, slot:slot + self.feature_dim] = f.squeeze(0).squeeze(-1).t()
            slot += self.feature_dim

        # Sparse levels – vectorised version
        for table, res, table_size in zip(self.hash_tables, self.hash_resolutions, self.hash_sizes):
            pos = coords * (res - 1)
            pos_floor = pos.floor().long()
            x0, y0 = pos_floor[:, 0], pos_floor[:, 1]
            x1 = (x0 + 1).clamp(max=res - 1)
            y1 = (y0 + 1).clamp(max=res - 1)
            wx = (pos[:, 0] - x0.float()).unsqueeze(1)   # (N,1)
            wy = (pos[:, 1] - y0.float()).unsqueeze(1)

            # 1. Stack corner coordinates
            corners = torch.stack([
                torch.stack([x0, y0], dim=1),
                torch.stack([x1, y0], dim=1),
                torch.stack([x0, y1], dim=1),
                torch.stack([x1, y1], dim=1)
            ], dim=1)   # (N, 4, 2)

            # 2. Vectorised hash
            hash_idx = ((corners[:, :, 0] * self.primes[0]) ^ 
                        (corners[:, :, 1] * self.primes[1])) % table_size

            # 3. Single gather
            gathered = table[hash_idx.long()]   # (N, 4, feat)

            # 4. Bilinear interpolation
            wx_ = wx.unsqueeze(-1)   # (N,1,1)
            wy_ = wy.unsqueeze(-1)
            f0 = torch.lerp(gathered[:, 0], gathered[:, 1], wx_)
            f1 = torch.lerp(gathered[:, 2], gathered[:, 3], wx_)
            feature = torch.lerp(f0, f1, wy_)   # (N, feat)

            out[:, slot:slot + self.feature_dim] = feature
            slot += self.feature_dim

        return torch.sigmoid(self.decoder(out))

In [12]:
def train_and_evaluate(config, dataset):
    model_name = config["name"]
    print(f"\n=== Starting: {model_name} ===")
    
    # Instant-NGP Multi-Res Grid
    model = OptimizedHashGrid2D(
        num_levels=config["levels"],
        base_res=16,
        max_res=config["max_res"],
        feature_dim=config["feat_dim"],
        log2_hashmap_size=config["log2"] # Add this!
    ).to(device)
    
    input_data = dataset["coords_raw"]
        
    optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    
    # Cosine Annealing Scheduler (Decays LR smoothly to 1e-6)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"], eta_min=1e-6)
    criterion = nn.MSELoss()
    
    # Automatic Mixed Precision Scaler
    scaler = torch.amp.GradScaler('cuda')
    
    batch_size = config["batch_size"]
    num_pixels = dataset["num_pixels"]
    
    model.train()
    pbar = tqdm(range(config["epochs"]), desc=model_name)
    
    for epoch in pbar:
        # Pre-shuffled Array (Done instantly on GPU)
        indices = torch.randperm(num_pixels, device=device)
        
        epoch_loss = 0.0
        batches = 0
        
        # Sequential Slicing (Zero GPU memory reallocation)
        for i in range(0, num_pixels, batch_size):
            batch_idx = indices[i : i + batch_size]
            b_coords = input_data[batch_idx]
            b_colors = dataset["colors"][batch_idx]
            
            optimizer.zero_grad(set_to_none=True) # Slightly faster than standard zero_grad
            
            # AMP Forward Pass
            with torch.amp.autocast('cuda'):
                pred = model(b_coords)
                loss = criterion(pred, b_colors)
                
            # AMP Backward Pass
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            epoch_loss += loss.detach()
            batches += 1
            
        scheduler.step()
        
        # Update UI every 50 epochs (Avoids stalling the pipeline)
        if epoch % 50 == 0:
            avg_loss = epoch_loss / batches
            psnr = -10.0 * torch.log10(avg_loss).item()
            current_lr = scheduler.get_last_lr()[0]
            pbar.set_postfix({"PSNR": f"{psnr:.2f}", "LR": f"{current_lr:.1e}"})

    # --- EVALUATION STAGE ---
    model.eval()
    start_time = time.time()
    predicted_colors = []
    
    # Chunked Inference with AMP
    with torch.no_grad(), torch.amp.autocast('cuda'):
        chunk_size = 65536 
        for i in range(0, num_pixels, chunk_size):
            chunk = input_data[i : i + chunk_size]
            predicted_colors.append(model(chunk))
            
        full_pred = torch.cat(predicted_colors, dim=0)
        
    torch.cuda.synchronize()
    latency_ms = (time.time() - start_time) * 1000

    # Save Output Image
    pred_np = full_pred.float().cpu().numpy().reshape(dataset["h"], dataset["w"], 3)
    pred_img_uint8 = (pred_np * 255).clip(0, 255).astype(np.uint8)
    os.makedirs("images", exist_ok=True)
    cv2.imwrite(f"images/{model_name}.png", cv2.cvtColor(pred_img_uint8, cv2.COLOR_RGB2BGR))

    # --- EXTREME STORAGE MINIMIZATION ---
    os.makedirs("models", exist_ok=True)
    pth_path = f"models/{model_name}.pth"
    zip_path = f"models/{model_name}.zip"
    
    # Convert weights to FP16 before saving (Halves the size immediately)
    model.half()
    torch.save(model.state_dict(), pth_path)
    
    # Compress using LZMA (Aggressive compression for numbers)
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_LZMA) as zipf:
        zipf.write(pth_path, arcname=f"{model_name}.pth")
    size_kb = os.path.getsize(zip_path) / 1024.0

    # Compute Metrics
    engine = MetricsEngine(device)
    metrics = engine.compute_all(dataset["original_np"], pred_img_uint8)
    
    row = {"Method": model_name, "Size_KB": round(size_kb, 2), "Latency_ms": round(latency_ms, 2)}
    row.update(metrics)
    
    # --- CLEAN SLATE PROTOCOL (VRAM PURGE) ---
    del model
    del optimizer
    del scheduler
    del scaler
    del full_pred
    del predicted_colors
    del criterion
    gc.collect()
    torch.cuda.empty_cache()
    
    return row

In [ ]:

#torch.cuda.empty_cache()
#batchsize = dataset["num_pixels"] 

batchsize = 65536/4 # The sweet spot for throughput AND optimizer steps

EXPERIMENTS = [
    {
        "name": "Hash_Q30_Vector", 
        "levels": 12,       
        "max_res": 1024,    
        "feat_dim": 2, 
        "log2": 13,         
        "epochs": 400, 
        "batch_size": batchsize, 
        "lr": 1e-2
    },
    {
        "name": "Hash_Q70_Vector", 
        "levels": 14, 
        "max_res": 2048, 
        "feat_dim": 2, 
        "log2": 14,         
        "epochs": 1000, 
        "batch_size": batchsize, 
        "lr": 1e-2
    }
]

os.makedirs("results", exist_ok=True) 
all_neural_results = []

for config in EXPERIMENTS: 
    result = train_and_evaluate(config, dataset)
    all_neural_results.append(result)
    
    # Save intermediate in case of crash
    pd.DataFrame(all_neural_results).to_csv("results/neural_metrics_temp.csv", index=False)
    print(f"Finished {config['name']} | VRAM Reset to: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

#Final Save
df_neural = pd.DataFrame(all_neural_results)
df_neural.to_csv("results/neural_metrics.csv", index=False) 
print("\nAll experiments completed and fully optimized!") 
display(df_neural)


=== Starting: Hash_Q30_Vector ===


Hash_Q30_Vector:   0%|          | 0/400 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 32.00 GiB. GPU 0 has a total capacity of 6.00 GiB of which 4.37 GiB is free. Of the allocated memory 544.47 MiB is allocated by PyTorch, and 47.53 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import torch
import torch.profiler
import numpy as np

# ── Minimal single-epoch trace ──────────────────────────────────────────────
# Uses your actual model + data from the existing `dataset` and `EXPERIMENTS[0]` config.
batchsize = dataset["num_pixels"]
EXPERIMENTS = [
    {
        "name": "Hash_Q30_Vector", 
        "levels": 12,       
        "max_res": 1024,    
        "feat_dim": 2, 
        "log2": 13,         
        "epochs": 400, 
        "batch_size": batchsize, 
        "lr": 1e-2
    },
    {
        "name": "Hash_Q70_Vector", 
        "levels": 14, 
        "max_res": 2048, 
        "feat_dim": 2, 
        "log2": 14,         
        "epochs": 1000, 
        "batch_size": batchsize, 
        "lr": 1e-2
    }
]

config  = EXPERIMENTS[0]
model   = OptimizedHashGrid2D(
    num_levels=config["levels"], base_res=16, max_res=config["max_res"],
    feature_dim=config["feat_dim"], log2_hashmap_size=config["log2"]
).to(device)

optimizer  = torch.optim.Adam(model.parameters(), lr=config["lr"])
scaler     = torch.amp.GradScaler('cuda')
criterion  = torch.nn.MSELoss()
input_data = dataset["coords_raw"]
num_pixels = dataset["num_pixels"]
batch_size = config["batch_size"]

# ── 1. TIMING BREAKDOWN — measures exactly where time goes ──────────────────
print("\n=== TIMING BREAKDOWN (single epoch) ===")
torch.cuda.synchronize()

indices       = torch.randperm(num_pixels, device=device)
t_data_total  = 0.0
t_fwd_total   = 0.0
t_bwd_total   = 0.0
t_step_total  = 0.0

for i in range(0, min(num_pixels, batch_size * 10), batch_size):   # first 10 batches only
    # --- Data fetch ---
    torch.cuda.synchronize(); t0 = torch.cuda.Event(enable_timing=True); t0.record()
    batch_idx = indices[i : i + batch_size]
    b_coords  = input_data[batch_idx]
    b_colors  = dataset["colors"][batch_idx]
    torch.cuda.synchronize(); t1 = torch.cuda.Event(enable_timing=True); t1.record()

    # --- Forward ---
    optimizer.zero_grad(set_to_none=True)
    t2 = torch.cuda.Event(enable_timing=True); t2.record()
    with torch.amp.autocast('cuda'):
        pred = model(b_coords)
        loss = criterion(pred, b_colors)
    t3 = torch.cuda.Event(enable_timing=True); t3.record()

    # --- Backward ---
    t4 = torch.cuda.Event(enable_timing=True); t4.record()
    scaler.scale(loss).backward()
    t5 = torch.cuda.Event(enable_timing=True); t5.record()

    # --- Optimizer step ---
    t6 = torch.cuda.Event(enable_timing=True); t6.record()
    scaler.step(optimizer); scaler.update()
    t7 = torch.cuda.Event(enable_timing=True); t7.record()

    torch.cuda.synchronize()
    t_data_total += t0.elapsed_time(t1)
    t_fwd_total  += t2.elapsed_time(t3)
    t_bwd_total  += t4.elapsed_time(t5)
    t_step_total += t6.elapsed_time(t7)

batches = min(num_pixels, batch_size * 10) // batch_size
print(f"  Data slice    : {t_data_total/batches:.3f} ms/batch")
print(f"  Forward pass  : {t_fwd_total/batches:.3f} ms/batch")
print(f"  Backward pass : {t_bwd_total/batches:.3f} ms/batch")
print(f"  Optimizer step: {t_step_total/batches:.3f} ms/batch")
total = t_data_total + t_fwd_total + t_bwd_total + t_step_total
print(f"  Total measured: {total/batches:.3f} ms/batch")

# ── 2. GPU UTILISATION — samples SM activity ────────────────────────────────
print("\n=== GPU UTILISATION (torch.profiler, 5 steps) ===")
model.train()
indices = torch.randperm(num_pixels, device=device)

with torch.profiler.profile(
    activities=[
        torch.profiler.ProfilerActivity.CPU,
        torch.profiler.ProfilerActivity.CUDA,
    ],
    schedule=torch.profiler.schedule(wait=1, warmup=1, active=3),
    on_trace_ready=torch.profiler.tensorboard_trace_handler('./prof_trace'),
    record_shapes=True,
    with_stack=False,
) as prof:
    for i in range(0, batch_size * 5, batch_size):
        batch_idx = indices[i : i + batch_size]
        b_coords  = input_data[batch_idx]
        b_colors  = dataset["colors"][batch_idx]
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            pred = model(b_coords)
            loss = criterion(pred, b_colors)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        prof.step()

# Human-readable table — top 15 CUDA ops by self time
print(prof.key_averages().table(
    sort_by="cuda_time_total", row_limit=15
))

# ── 3. MEMORY SNAPSHOT ──────────────────────────────────────────────────────
print(f"\n=== VRAM SNAPSHOT ===")
print(f"  Allocated : {torch.cuda.memory_allocated()/1024**2:.1f} MB")
print(f"  Reserved  : {torch.cuda.memory_reserved()/1024**2:.1f} MB")
print(f"  Peak alloc: {torch.cuda.max_memory_allocated()/1024**2:.1f} MB")
alloc = torch.cuda.memory_allocated()/1024**2
reserved = torch.cuda.memory_reserved()/1024**2
print(f"  Fragmentation ratio: {1 - alloc/reserved:.2%}  (>30% = bad)")


=== TIMING BREAKDOWN (single epoch) ===
  Data slice    : 21.642 ms/batch
  Forward pass  : 597.887 ms/batch
  Backward pass : 179.842 ms/batch
  Optimizer step: 45.278 ms/batch
  Total measured: 844.649 ms/batch

=== GPU UTILISATION (torch.profiler, 5 steps) ===
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us      44.240ms 